# Pipeline QA & exploration

Runs the `sme_pipeline` package and inspects its output. This notebook deliberately holds **no parsing or joining logic of its own** - that all lives in the package, so what is checked here is the same code that produces `data/processed/`.

Structural findings that came out of the original exploration are now recorded in `DATA_PIPELINE_SPEC.md`.

In [1]:
import pandas as pd

from sme_pipeline import config, pipeline, transform

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

result = pipeline.run(write=False, verbose=False)
firm, individual, universe = result.firm_table, result.individual_table, result.universe
{"firm": firm.shape, "individual": individual.shape, "universe": universe.shape}

{'firm': (61438, 8), 'individual': (3174, 9), 'universe': (37177, 5)}

## Join health

Match rate = share of firm rows that found an `enterprise_count` denominator. The shortfalls are data-availability boundaries documented in the spec (time coverage, NACE aggregates), not defects - but a rate *dropping* below its documented baseline signals a regression.

In [2]:
transform.join_report(firm)

,dataset,rows,matched,match_rate
0,isoc_e_dii,795,300,0.377358
1,isoc_eb_ai,5172,3403,0.657966
2,isoc_eb_ain2,44319,13744,0.310115
3,isoc_eb_bd,1229,0,0.000000
4,isoc_eb_bdn2,8513,0,0.000000
5,isoc_eb_das,1410,705,0.500000


In [3]:
# Why rows go unmatched: indicator years outside the universe's 2021-2024 window.
coverage = pd.DataFrame(
    [(code, ", ".join(sorted(df["time"].unique()))) for code, df in result.tidy.items()],
    columns=["dataset", "years"],
)
coverage

,dataset,years
0,sbs_sc_ovw,"2021, 2022, 2023, 2024"
1,isoc_eb_ai,"2021, 2023, 2024, 2025"
2,isoc_eb_bd,"2016, 2018, 2020"
3,isoc_eb_das,"2023, 2025"
4,isoc_e_dii,"2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022..."
5,isoc_eb_ain2,"2021, 2023, 2024, 2025"
6,isoc_eb_bdn2,"2016, 2018, 2020"
7,isoc_ai_iaiu,2025


## The headline story: SME vs large AI adoption

`E_AI_TTM|PC_ENT` = share of enterprises using any AI technology.

In [4]:
gap = firm[
    (firm["indicator"] == "E_AI_TTM|PC_ENT")
    & (firm["time"] == "2024")
    & (firm["size_emp"].isin(config.SME_CLASSES + [config.LARGE_CLASS]))
]
gap.pivot_table(index="geo", columns="size_emp", values="value")

size_emp,LARGE_GE250,MEDIUM_50_249,SMALL_10_49,SME_10_249
geo,,,,
DE,24.31,13.64,8.16,9.06
EU27_2020,21.44,10.38,5.76,6.44
IT,19.76,8.18,3.70,4.18


In [5]:
# Percentages weighted by the universe -> estimated absolute enterprise counts.
# These are estimates: see DATA_PIPELINE_SPEC.md Caveat 3 on the NACE scope gap.
weighted = gap[gap["enterprise_count"].notna()].assign(
    enterprises_using_ai=lambda d: (d["value"] / 100 * d["enterprise_count"]).round()
)
weighted[["geo", "size_emp", "value", "enterprise_count", "enterprises_using_ai"]].sort_values(["geo", "size_emp"])

,geo,size_emp,value,enterprise_count,enterprises_using_ai
4177,DE,LARGE_GE250,24.31,15096.0,3670.0
3111,DE,MEDIUM_50_249,13.64,65590.0,8946.0
22,DE,SMALL_10_49,8.16,420678.0,34327.0
1030,DE,SME_10_249,9.06,486268.0,44056.0
4173,EU27_2020,LARGE_GE250,21.44,54871.0,11764.0
3107,EU27_2020,MEDIUM_50_249,10.38,250670.0,26020.0
18,EU27_2020,SMALL_10_49,5.76,1569947.0,90429.0
1026,EU27_2020,SME_10_249,6.44,1820617.0,117248.0
4181,IT,LARGE_GE250,19.76,4899.0,968.0
3115,IT,MEDIUM_50_249,8.18,26891.0,2200.0


## Individual-level table

`ind_type` packs sex, age, education, labour status and citizenship into one dimension. `normalize.decode_ind_type` unpacks the demographic subset; non-demographic codes (occupation, urbanisation) correctly decode to all-null.

In [6]:
print("decoded coverage:")
for column in ["sex", "age", "education"]:
    print(f"  {column}: {individual[column].notna().mean():.0%}")

individual[["ind_type", "sex", "age", "education"]].drop_duplicates().head(15)

decoded coverage:
  sex: 39%
  age: 65%
  education: 29%


,ind_type,sex,age,education
0,IND_TOTAL,NaN,NaN,NaN
33,Y16_17,NaN,Y16_17,NaN
66,Y16_19,NaN,Y16_19,NaN
99,Y16_24,NaN,Y16_24,NaN
132,Y16_29,NaN,Y16_29,NaN
165,Y20_24,NaN,Y20_24,NaN
198,Y25_29,NaN,Y25_29,NaN
231,Y25_34,NaN,Y25_34,NaN
264,Y25_54,NaN,Y25_54,NaN
297,Y25_64,NaN,Y25_64,NaN


## Readiness gap: individuals vs their employers

Generative AI use among people, against AI use among enterprises, same year.

In [7]:
people = individual[
    (individual["ind_type"] == "IND_TOTAL") & (individual["time"] == "2024")
][["geo", "indicator", "value"]]

firms = firm[
    (firm["indicator"] == "E_AI_TTM|PC_ENT")
    & (firm["time"] == "2024")
    & (firm["size_emp"] == "SME_10_249")
][["geo", "value"]].rename(columns={"value": "sme_ai_pct"})

people.merge(firms, on="geo", how="left").head(20)

,geo,indicator,value,sme_ai_pct
